# 06 - Prepare Per-Crop Disease Datasets

Stage 6: Build per-crop disease datasets from manifest.csv (produced by
01_prepare_crop_dataset.py, which already recorded disease_raw + environment
per image).

WHY THIS NEEDS YOUR INPUT, NOT JUST A RUN:
Your raw folder names use inconsistent taxonomies across environments (e.g.
Potato Closed uses "Early Blight"/"Late Blight" while Potato Uncontrolled
uses broad categories like "Fungi"/"Bacteria" -- these are NOT the same
labeling system and should not be blindly merged). CANONICAL_DISEASE_MAP
below is a STARTER mapping fixing only unambiguous typos/casing. Review
and extend it yourself, crop by crop, before trusting the output --
especially the flagged crops.

Install deps:
    pip install pandas scikit-learn tqdm --break-system-packages

## Imports & Configuration

In [ ]:
import shutil
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm

MANIFEST_PATH = Path(r"Z:\Projects\Smart-Farming\models\csvs\manifest.csv")
OUT_DIR = Path(r"Z:\Projects\Smart-Farming\Datasets\disease_dataset")
MIN_CLASS_COUNT = 30  # classes with fewer images than this are dropped, not trained on

# ---- Pest-damage folders excluded from disease classification -----------
# Confirmed: these are symptoms of pests, not diseases. Excluded from
# training now -- reserved for the future YOLOv8 pest-detection module.
# REMINDER (as requested): don't lose track of these -- they still need a
# home in the Pest Detection Service down the line.
PEST_CLASSES_TO_EXCLUDE = {
    "A", "AW", "SM", "T", "LM",
}

# ---- Canonical mapping: crop -> {raw_label: canonical_label} ------------
# Confirmed final taxonomy per crop. Correct-spelling canonical names only.
# Pest-symptom rows above are still mapped here (for consistent spelling)
# but get filtered out by PEST_CLASSES_TO_EXCLUDE before training.
CANONICAL_DISEASE_MAP = {
    "Cotton": {
        "ALS": "Alternaria Leaf Spot",
        "BB": "Bacterial Blight",
        "FW": "Fusarium Wilt",
        "HL": "Healthy",
        "H": "Healthy",
        "VW": "Verticillium Wilt",
        "CV": "Curl Virus",
        "PM": "Powdery Mildew",
        "TS": "Target Spot",
        "A": "Aphid",          # excluded (pest symptom)
        "AW": "Army Worm",    # excluded (pest symptom)
    },
    "Groundnut": {
        # Confirmed: early + late leaf spot combined into one "Leaf Spot"
        # class (including Field Closeup's already-merged folder and
        # early_rust, which folds into "Rust" since no separate
        # "Early Rust" class was requested).
        "ELS": "Leaf Spot",
        "LLS": "Leaf Spot",
        "LSEaL": "Leaf Spot",
        "ALS": "Althernaria Leaf Spot",
        "Ros": "Rosette",
        "HL": "Healthy",
        "H": "Healthy",
        "ND": "Nutrition Deficiency",
        "R": "Rust",
        "ER": "Rust",
    },
    "Pepper Bell": {
        "BS": "Bacterial Spot",
        "CLS": "Cercospora Leaf Spot",
        "H": "Healthy",
        "LC": "Leaf Curl",
        "ND": "Nutrition Deficiency",
        "PM": "Powdery Mildew",
        "BER": "Blossom End Rot",
        # Kept as its own class per your note that "Burn" is often
        # mislabeled/grouped with Blossom End Rot or Nutrient Deficiency
        # depending on the dataset creator -- only 5 images, so
        # MIN_CLASS_COUNT will prune it automatically unless you add more.
        "B": "Burn",
        "E": "Edema",
        "A": "Aphid",              # excluded (pest symptom)
        "LM": "Leaf Miner",   # excluded (pest symptom)
        "SM": "Spider Mite",   # excluded (pest symptom)
        "T": "Thrips",             # excluded (pest symptom)
        # NOTE: "Mosaic Virus" was on your confirmed list but there's no
        # matching raw folder in your current Pepper Bell directories --
        # if you have images for it, add the raw folder name here.
    },
    "Potato": {
        # Confirmed: keep Closed Environment's specific names AND
        # Uncontrolled's broad categories as their own separate classes
        # (no forced merge), except Late Blight = Phytopthora per your
        # instruction.
        "EB": "Early Blight",
        "LB": "Late Blight ",
        "Phy": "Late Blight ",   # merged per your instruction
        "H": "Healthy",
        "B": "Bacteria",
        "F": "Fungi",
        "V": "Virus",
        "N": "Nematode",
        "P": "Pest",
    },
    "Tomato": {
        "BS": "Bacterial Spot",
        "EB": "Early Blight",
        "LB": "Late Blight",
        "H": "Healthy",
        "ML": "Mold Leaf",
        "MV": "Mosaic Virus",
        "S": "Septoria",
        "YCV": "Yellow Curl Virus",
        "DoPM": "Defiency of Phosphor Magnesium",
        "BL": "Burn Leaf",
        "WL": "Wealth Leaf",   # unclear label -- verify what this actually means
        "LM": "Leaf Miner",     # excluded (pest symptom)
        "SM": "Spider Mite",   # excluded (pest symptom)
    },
}

## `build_disease_manifest`

In [ ]:
def build_disease_manifest(crop_name):
    df = pd.read_csv(MANIFEST_PATH)
    df = df[df["crop"] == crop_name].copy()

    crop_map = CANONICAL_DISEASE_MAP.get(crop_name, {})
    df["disease_raw"] = df["disease_raw"].astype(str).str.strip()

    # Exclude pest-damage folders
    before = len(df)
    df = df[~df["disease_raw"].isin(PEST_CLASSES_TO_EXCLUDE)]
    excluded = before - len(df)
    if excluded:
        print(f"[{crop_name}] Excluded {excluded} pest-damage images (reserved for YOLOv8 module)")

    # Apply canonical mapping; anything unmapped passes through as-is and
    # gets flagged so you notice it
    unmapped = set(df["disease_raw"]) - set(crop_map.keys())
    if unmapped:
        print(f"[{crop_name}] WARNING: no canonical mapping for: {sorted(unmapped)}"
              f" -- these will be used as their raw folder names. Add them to"
              f" CANONICAL_DISEASE_MAP if that's not what you want.")

    df["disease"] = df["disease_raw"].map(lambda x: crop_map.get(x, x))
    return df

## `prune_rare_classes`

In [ ]:
def prune_rare_classes(df, min_count=MIN_CLASS_COUNT):
    counts = df["disease"].value_counts()
    rare = counts[counts < min_count].index.tolist()
    if rare:
        print(f"Dropping rare classes (< {min_count} images): {rare}")
    return df[~df["disease"].isin(rare)]

## `split_and_materialize`

In [ ]:
def split_and_materialize(crop_name, df, train_size=0.7, val_size=0.15, test_size=0.15, seed=42):
    import os
    import sys
    locked_train = df[df["environment"] == "derived"]
    splittable = df[df["environment"] != "derived"]

    strat_key = splittable["disease"]
    train_df, temp_df = train_test_split(
        splittable, train_size=train_size, stratify=strat_key, random_state=seed
    )
    train_df = pd.concat([train_df, locked_train], ignore_index=True)

    remaining_key = temp_df["disease"]
    relative_val = val_size / (val_size + test_size)
    val_df, test_df = train_test_split(
        temp_df, train_size=relative_val, stratify=remaining_key, random_state=seed
    )

    for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"{crop_name}/{split_name}"):
            dest_dir = OUT_DIR / crop_name.replace(" ", "_") / split_name / row["disease"]
            dest_dir.mkdir(parents=True, exist_ok=True)
            src = Path(row["filepath"])
            dest = dest_dir / f"img_{abs(hash(str(src)))}{src.suffix}"
            
            src_str = str(src.resolve().absolute())
            dest_str = str(dest.resolve().absolute())
            if sys.platform == "win32":
                if not src_str.startswith("\\\\?\\"):
                    src_str = "\\\\?\\" + src_str
                if not dest_str.startswith("\\\\?\\"):
                    dest_str = "\\\\?\\" + dest_str
            shutil.copy2(src_str, dest_str)

    print(f"[{crop_name}] Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

## `prepare_crop_disease_dataset`

In [ ]:
def prepare_crop_disease_dataset(crop_name):
    print(f"\n=== {crop_name} ===")
    df = build_disease_manifest(crop_name)
    print(df.groupby("disease").size().sort_values(ascending=False))
    df = prune_rare_classes(df)
    
    # ---- Save final processed manifest for this crop ----
    csvs_output_dir = Path(r"Z:\Projects\Smart-Farming\models\csvs")
    csvs_output_dir.mkdir(parents=True, exist_ok=True)
    
    safe_crop_name = crop_name.lower().replace(" ", "_")
    manifest_output_path = csvs_output_dir / f"{safe_crop_name}_disease_manifest.csv"
    df.to_csv(manifest_output_path, index=False)
    print(f"Saved disease manifest to:\n{manifest_output_path.resolve()}")
    
    split_and_materialize(crop_name, df)

## Run

In [ ]:
for crop in ["Cotton", "Groundnut", "Pepper Bell", "Potato", "Tomato"]:
    prepare_crop_disease_dataset(crop)